# 12 - Multiple Cross-Model Readers: Prefill

Add Llama-3.2-3B and Mistral-7B as additional cross-model readers.
Run cross_prefill and paraphrase_cross for each using existing Qwen3-4B COTs and paraphrases.

Tests whether legibility is universal or specific to the Qwen-Gemma pair.

In [5]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Already up to date.


In [6]:
import json
import gc, torch
from lib.data import load_gsm8k
from lib.prefill import run_prefill_condition

dataset = load_gsm8k()

# Load verbatim COTs
cot_lookup = {}
for p in COT_CACHE.glob("*.json"):
    r = json.loads(p.read_text())
    cot_lookup[r["problem_id"]] = r["cot_text"]

# Load light paraphrases
para_lookup = {}
for p in PARAPHRASE_CACHE.glob("paraphrase_light_*.json"):
    r = json.loads(p.read_text())
    para_lookup[r["problem_id"]] = r["paraphrased_cot"]

print(f"COTs: {len(cot_lookup)}, Paraphrases: {len(para_lookup)}")

def reader_tag(name):
    return name.split("/")[-1].lower().replace("-", "_").replace(".", "")

COTs: 1319, Paraphrases: 1319


In [ ]:
from vllm import LLM, SamplingParams

for reader_model in CROSS_READER_MODELS:
    tag = reader_tag(reader_model)
    print(f"\n{'='*60}")
    print(f"Reader: {reader_model} ({tag})")
    print(f"{'='*60}")

    print(f"Loading {reader_model}...")
    try:
        llm = LLM(model=reader_model, dtype="bfloat16", max_model_len=4096)
    except Exception as e:
        print(f"Failed to load: {e}")
        continue

    # Cross prefill (verbatim COT)
    run_prefill_condition(llm, f"cross_prefill_{tag}", reader_model, dataset, cot_lookup)

    # Paraphrase cross (light paraphrased COT)
    run_prefill_condition(llm, f"paraphrase_cross_{tag}", reader_model, dataset, para_lookup)

    # No-COT baseline for this reader
    from lib.prompts import build_no_cot_messages
    from lib.data import extract_predicted_answer
    from transformers import AutoTokenizer
    from tqdm import tqdm

    nc_cond = f"no_cot_{tag}"
    nc_done = set()
    for p in PREFILL_CACHE.glob(f"{nc_cond}_*.json"):
        # Problem ID is always the last _-separated segment before .json
        try:
            nc_done.add(int(p.stem.rsplit("_", 1)[-1]))
        except ValueError:
            pass
    nc_todo = [ex for ex in dataset if ex["problem_id"] not in nc_done]
    print(f"No-COT ({tag}): {len(nc_done)} done, {len(nc_todo)} remaining")

    if nc_todo:
        tok = AutoTokenizer.from_pretrained(reader_model)
        sampling = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_ANSWER_TOKENS)
        for i in tqdm(range(0, len(nc_todo), CHUNK_SIZE), desc=f"No-COT ({tag})"):
            chunk = nc_todo[i:i + CHUNK_SIZE]
            prompts = [
                tok.apply_chat_template(
                    build_no_cot_messages(ex["question"]),
                    tokenize=False, add_generation_prompt=True
                ) for ex in chunk
            ]
            outputs = llm.generate(prompts, sampling)
            for ex, output in zip(chunk, outputs):
                generated = output.outputs[0].text.strip()
                result = {
                    "problem_id": ex["problem_id"],
                    "condition": nc_cond,
                    "model_used": reader_model,
                    "prefill_text": None,
                    "predicted_answer": extract_predicted_answer(generated),
                    "gold_answer": ex["gold_answer"],
                    "generated_tokens": generated,
                    "error": None,
                }
                (PREFILL_CACHE / f"{nc_cond}_{ex['problem_id']}.json").write_text(json.dumps(result))

    del llm
    gc.collect()
    torch.cuda.empty_cache()
    print(f"{reader_model} complete and unloaded.")

In [4]:
# Summary
for reader_model in CROSS_READER_MODELS:
    tag = reader_tag(reader_model)
    for cond in [f"cross_prefill_{tag}", f"paraphrase_cross_{tag}", f"no_cot_{tag}"]:
        n = len(list(PREFILL_CACHE.glob(f"{cond}_*.json")))
        print(f"{cond:45s}: {n} results")

cross_prefill_gemma_3_4b_it                  : 1319 results
paraphrase_cross_gemma_3_4b_it               : 1319 results
no_cot_gemma_3_4b_it                         : 1319 results
cross_prefill_llama_32_3b_instruct           : 0 results
paraphrase_cross_llama_32_3b_instruct        : 0 results
no_cot_llama_32_3b_instruct                  : 0 results
cross_prefill_mistral_7b_instruct_v03        : 1319 results
paraphrase_cross_mistral_7b_instruct_v03     : 1319 results
no_cot_mistral_7b_instruct_v03               : 1319 results
